In [39]:
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

In [40]:
firms_data = [
    ("AAPL", "Apple", 35),
    ("ABBV", "Abbvie", 28),
    ("ABT", "Abbott Lab.", 50),
    ("ACN", "Accenture", 67),
    ("ADBE", "Adobe", 73),
    ("AIG", "Ame Inter", 63),
    ("AMD", "Adv Micro Dev", 36),
    ("AMGN", "Amgen", 28),
    ("AMT", "American Tower", 48),
    ("AMZN", "Amazon", 73),
    ("AVGO", "Broadcom", 36),
    ("AXP", "Amex", 60),
    ("BA", "Boeing", 37),
    ("BAC", "Bank of Am", 60),
    ("BH", "Biglari Holdings", 58),
    ("BK", "Bank of NY", 60),
    ("BKNG", "Booking", 73),
    ("BLK", "Blackrock", 62),
    ("BMY", "Bristol-Myers", 28),
    ("C", "Citigroup", 60),
    ("CAT", "Caterpillar", 35),
    ("CHTR", "Charter Comm", 48),
    ("CL", "Colgate Palmo", 28),
    ("CMCSA", "Comcast", 48),
    ("COF", "Capital One", 60),
    ("COP", "Conocophillips", 13),
    ("COST", "Costco", 53),
    ("CRM", "Salesforce", 73),
    ("CSCO", "Cisco Sys", 36),
    ("CVS", "C V S Health", 59),
    ("CVX", "Chevron", 13),
    ("D", "Dominion En", 49),
    ("DE", "Deere", 35),
    ("DHR", "Danaher", 50),
    ("DIS", "Disney", 73),
    ("DUK", "Duke Energy", 49),
    ("EMR", "Emerson", 35),
    ("F", "Ford", 37),
    ("FDX", "Fedex", 45),
    ("GD", "Gen Dynamics", 37),
    ("GE", "Gen Electric", 35),
    ("GILD", "Gilead", 28),
    ("GM", "General Motors", 37),
    ("GOOG", "Alphabet", 73),
    ("GOOGL", "Alphabet", 73),
    ("GS", "Goldman Sachs", 62),
    ("HD", "Home Depot", 52),
    ("HON", "Honeywell Int", 50),
    ("IBM", "IBM", 73),
    ("INTC", "Intel", 36),
    ("INTU", "Intuit", 73),
    ("JNJ", "Johnson&J", 28),
    ("JPM", "Jpmorgan", 60),
    ("KO", "Coca Cola", 20),
    ("LLY", "Lilly Eli", 28),
    ("LMT", "Lockheed Mar", 37),
    ("LOW", "Lowes", 52),
    ("MA", "Mastercard", 73),
    ("MCD", "Mcdonalds", 58),
    ("MDLZ", "Mondelez Int", 20),
    ("MDT", "Medtronic", 38),
    ("MET", "Metlife", 63),
    ("META", "Meta", 73),
    ("MMM", "3M", 50),
    ("MO", "Altria Group", 21),
    ("MRK", "Merck", 28),
    ("MS", "Morgan Stanley", 62),
    ("MSFT", "Microsoft", 73),
    ("NEE", "Nextera Energy", 49),
    ("NFLX", "Netflix", 78),
    ("NKE", "Nike", 30),
    ("NVDA", "Nvidia", 36),
    ("ORCL", "Oracle", 73),
    ("PEP", "Pepsico", 20),
    ("PFE", "Pfizer", 28),
    ("PG", "Procter Gamble", 28),
    ("PM", "Philip Morris", 21),
    ("QCOM", "Qualcomm", 36),
    ("RTX", "RTX", 37),
    ("SBUX", "Starbucks", 58),
    ("SCHW", "Schwab Charles", 62),
    ("SO", "Southern", 49),
    ("SPG", "Simon Property", 67),
    ("T", "AT&T", 48),
    ("TGT", "Target", 53),
    ("TMO", "Thermo Fisher", 38),
    ("TMUS", "T-Mobile", 48),
    ("TSLA", "Tesla", 37),
    ("TXN", "Texas Instru", 36),
    ("UNH", "Unitedhealth", 63),
    ("UNP", "Union Pacific", 40),
    ("UPS", "United Parcel", 45),
    ("USB", "US Bancorp", 60),
    ("V", "Visa", 73),
    ("VZ", "Verizon", 48),
    ("WFC", "Wells Fargo", 60),
    ("WMT", "Walmart", 53),
    ("XOM", "Exxon Mobil", 29),
]

firms = pd.DataFrame(firms_data, columns=["ticker", "name_table", "sic2"])



In [41]:
TIINGO_API_KEY = "f453c0df7a8ed7e841c849e698b714dd2309bdff"


def download_tiingo_one(ticker, start="2015-01-01", end="2024-12-31"):
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Token {TIINGO_API_KEY}",
    }

    params = {
        "startDate": start,
        "endDate": end,
        "format": "json",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if len(data) == 0:
        raise ValueError(f"No data for {ticker}")

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df["ticker"] = ticker

    return df

In [43]:
all_prices = []
failed = []

step = 50
nb_batches = 2
batch_idx = 0

time.sleep(3700)

for i in range(0, len(firms), step):
    batch = firms.iloc[i:i+step]

    for _, row in tqdm(batch.iterrows(), total=len(batch)):
        ticker = row["ticker"]

        try:
            tmp = download_tiingo_one(
                ticker=ticker,
                start="2014-12-31",
                end="2024-12-31"
            )

            tmp["name_table"] = row["name_table"]
            tmp["sic2"] = row["sic2"]

            all_prices.append(tmp)

            time.sleep(0.3)

        except Exception as e:
            print(f"Échec {ticker}: {e}")
            failed.append(ticker)

    prices_tiingo = pd.concat(all_prices, ignore_index=True)

    prices_tiingo = prices_tiingo.sort_values(["ticker", "date"]).reset_index(drop=True)

    if batch_idx != 1:
        time.sleep(3700)
        batch_idx = 1

prices_tiingo.to_csv('SP100_tickers.csv')
print("Tickers récupérés :", prices_tiingo["ticker"].nunique())
print("Tickers échoués :", failed)

prices_tiingo.head()

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/48 [00:00<?, ?it/s]

Tickers récupérés : 98
Tickers échoués : []


,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,ticker,name_table,sic2
0,2014-12-31,110.38,113.13,110.210,112.82,41403351,24.447725,25.056814,24.410073,24.988154,165613404,0.0,1.0,AAPL,Apple,35
1,2015-01-02,109.33,111.44,107.350,111.39,53204626,24.215164,24.682502,23.776620,24.671427,212818504,0.0,1.0,AAPL,Apple,35
2,2015-01-05,106.25,108.65,105.410,108.29,64285491,23.532985,24.064553,23.346936,23.984818,257141964,0.0,1.0,AAPL,Apple,35
3,2015-01-06,106.26,107.43,104.630,106.54,65797116,23.535199,23.794339,23.174176,23.597216,263188464,0.0,1.0,AAPL,Apple,35
4,2015-01-07,107.75,108.20,106.695,107.20,40105934,23.865215,23.964884,23.631546,23.743397,160423736,0.0,1.0,AAPL,Apple,35


In [69]:
prices_tiingo.to_csv("prices_tiingo.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 1) Charger le fichier
# ============================================================

# Si le fichier est dans le même dossier que ton notebook
prices = pd.read_csv("prices_tiingo.csv")

# Si besoin, adapte le chemin :
# prices = pd.read_csv(r"C:\Users\33761\Downloads\prices_tiingo.csv")


# ============================================================
# 2) Nettoyage
# ============================================================

prices["date"] = pd.to_datetime(prices["date"])
prices["ticker"] = prices["ticker"].astype(str)

prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

# Optionnel : commencer exactement en 2015
prices = prices[prices["date"] >= "2015-01-01"].copy()

# On garde une seule ligne par couple date/ticker
prices = prices.drop_duplicates(subset=["date", "ticker"], keep="last")


# ============================================================
# 3) Fonction pour créer un DataFrame wide SANS pivot
# ============================================================

def make_wide_dataframe(df, value_col):
    """
    Transforme un DataFrame long en DataFrame wide :
    - index = date
    - colonnes = ticker
    - valeurs = value_col

    Cette fonction évite pivot() et pivot_table().
    """

    all_dates = sorted(df["date"].unique())
    all_tickers = sorted(df["ticker"].unique())

    wide = pd.DataFrame(index=all_dates)
    wide.index.name = "date"

    for ticker in all_tickers:
        sub = df[df["ticker"] == ticker].copy()

        sub = sub.sort_values("date")
        sub = sub.drop_duplicates(subset="date", keep="last")

        series = sub.set_index("date")[value_col]

        wide[ticker] = series

    return wide.sort_index()


# ============================================================
# 4) DataFrames wide
# ============================================================
close_wide = make_wide_dataframe(prices, "close")
adj_close_wide = make_wide_dataframe(prices, "adjClose")


# ============================================================
# 5) Rendements
# ============================================================

returns_wide = adj_close_wide.pct_change()

log_returns_wide = np.log(adj_close_wide / adj_close_wide.shift(1))


In [90]:
close_wide.to_csv('SP100_tickers_close.csv')

In [91]:
adj_close_wide

,AAPL,ABBV,ABT,ACN,ADBE,AIG,AMD,AMGN,AMT,AMZN,...,TXN,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,24.215164,41.345517,36.005221,72.660224,72.340,42.949880,2.69,115.686114,75.750064,15.4260,...,39.309968,83.853807,92.328384,73.438729,30.237825,61.220381,25.677960,39.972715,23.174225,57.459123
2015-01-05,23.532985,40.567425,36.013240,71.433408,71.980,42.100221,2.66,114.311396,74.655652,15.1095,...,38.696209,82.472607,89.206920,71.968357,29.509365,59.869014,25.464706,38.876571,23.106779,55.886936
2015-01-06,23.535199,40.366627,35.604272,70.918145,70.530,41.503164,2.63,110.628600,74.458050,14.7645,...,38.060399,82.306197,87.362065,71.495976,29.118155,59.483239,25.721705,38.065425,23.284835,55.589829
2015-01-07,23.865215,41.998109,35.892955,72.406682,71.110,41.702183,2.58,114.492280,75.240859,14.9210,...,38.743987,83.146566,87.844685,72.161302,29.374465,60.280199,25.557663,38.291961,23.902634,56.153093
2015-01-08,24.782171,42.437354,36.630701,73.510816,72.915,41.518473,2.61,114.079865,75.940067,15.0230,...,39.376122,87.115435,91.137401,73.458688,29.590305,61.088709,26.105446,39.139646,24.407126,57.087740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,256.813347,171.042069,111.339886,348.896753,447.940,70.891911,126.29,254.439846,175.138341,229.0500,...,185.152044,489.975346,223.263336,116.475706,45.825234,317.744885,36.048537,69.738029,91.611814,101.926488
2024-12-26,257.628944,170.281882,111.834686,347.739006,450.160,71.378538,125.06,253.179624,174.446169,227.0500,...,184.468933,494.864450,223.729784,116.567957,45.994158,318.002529,36.193456,69.903631,91.720546,102.012704
2024-12-27,254.217364,169.151104,111.563031,343.638652,446.480,71.047631,125.19,252.669763,173.981561,223.7500,...,183.939762,493.741408,223.438254,116.337330,45.506156,315.772914,36.157226,69.270446,90.603570,102.003124


In [92]:
adj_close_wide.to_csv('SP100_tickers_adj_close.csv')

In [96]:
prices = pd.read_csv('SP100_tickers_close.csv', index_col=0)

In [97]:
prices

,AAPL,ABBV,ABT,ACN,ADBE,AIG,AMD,AMGN,AMT,AMZN,...,TXN,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,109.33,65.89,44.90,88.84,72.340,56.11,2.69,159.89,99.67,308.52,...,53.480,100.78,118.61,110.38,44.83,265.02,46.96,54.70,85.90,92.83
2015-01-05,106.25,64.65,44.91,87.34,71.980,55.00,2.66,157.99,98.23,302.19,...,52.645,99.12,114.60,108.17,43.75,259.17,46.57,53.20,85.65,90.29
2015-01-06,106.26,64.33,44.40,86.71,70.530,54.22,2.63,152.90,97.97,295.29,...,51.780,98.92,112.23,107.46,43.17,257.50,47.04,52.09,86.31,89.81
2015-01-07,107.75,66.93,44.76,88.53,71.110,54.48,2.58,158.24,99.00,298.42,...,52.710,99.93,112.85,108.46,43.55,260.95,46.19,52.40,88.60,90.72
2015-01-08,111.89,67.63,45.68,89.88,72.915,54.24,2.61,157.67,99.92,300.46,...,53.570,104.70,117.08,110.41,43.87,264.45,47.18,53.56,90.47,92.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,258.20,180.00,114.76,361.63,447.940,72.84,126.29,264.49,184.71,229.05,...,192.440,506.10,229.75,126.26,48.83,320.65,39.80,71.59,92.68,106.40
2024-12-26,259.02,179.20,115.27,360.43,450.160,73.34,125.06,263.18,183.98,227.05,...,191.730,511.15,230.23,126.36,49.01,320.91,39.96,71.76,92.79,106.49
2024-12-27,255.59,178.01,114.99,356.18,446.480,73.00,125.19,262.65,181.87,223.75,...,191.180,509.99,229.93,126.11,48.49,318.66,39.92,71.11,91.66,106.48


In [98]:
from marginal_model import fit_all, get_uniform_matrix, summary_table

# prices: votre DataFrame (dates en index, 98 tickers en colonnes)
results = fit_all(prices, verbose=True)

# Table de synthèse (reproduit l'Appendix C, Table 3, Panel B)
print(summary_table(results))

# Matrice des u_{i,t} — input direct pour la copula (étape suivante)
U = get_uniform_matrix(results)

  [  1/98] AAPL    α=0.119  κ=-0.119  β=0.940  η=3.1  λ=-0.077
  [  2/98] ABBV    α=0.109  κ=-0.016  β=0.795  η=3.5  λ=-0.087
  [  3/98] ABT     α=0.022  κ=0.077  β=0.877  η=4.8  λ=-0.092
  [  4/98] ACN     α=0.000  κ=0.220  β=0.838  η=4.3  λ=-0.083
  [  5/98] ADBE    α=0.042  κ=0.114  β=0.877  η=3.9  λ=-0.105
  [  6/98] AIG     α=0.118  κ=0.138  β=0.724  η=5.2  λ=-0.089
  [  7/98] AMD     α=0.096  κ=0.079  β=0.655  η=4.0  λ=+0.031
  [  8/98] AMGN    α=0.053  κ=0.107  β=0.742  η=4.1  λ=-0.017
  [  9/98] AMT     α=0.039  κ=0.061  β=0.904  η=5.8  λ=-0.077
  [ 10/98] AMZN    α=0.000  κ=0.111  β=0.944  η=2.6  λ=-0.045
  [ 11/98] AVGO    α=0.104  κ=-0.104  β=0.948  η=2.5  λ=-0.181
  [ 12/98] AXP     α=0.001  κ=0.150  β=0.911  η=4.1  λ=-0.062
  [ 13/98] BA      α=0.040  κ=0.060  β=0.910  η=4.5  λ=-0.020
  [ 14/98] BAC     α=0.019  κ=0.159  β=0.838  η=5.7  λ=+0.017
  [ 15/98] BH      α=0.303  κ=-0.047  β=0.637  η=4.0  λ=+0.057
  [ 16/98] BK      α=0.021  κ=0.090  β=0.886  η=4.7  λ=-0.096
  [ 

In [99]:
U

,AAPL,ABBV,ABT,ACN,ADBE,AIG,AMD,AMGN,AMT,AMZN,...,TXN,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-06,0.414389,0.322635,0.166061,0.228332,0.104810,0.175838,0.312404,0.017727,0.387555,0.248682,...,0.116832,0.403511,0.063416,0.285070,0.187497,0.752893,0.850412,0.111483,0.761057,0.350369
2015-01-07,0.736109,0.990846,0.719955,0.938951,0.648520,0.605227,0.211046,0.967910,0.775180,0.631603,...,0.851361,0.773551,0.624820,0.784152,0.711610,0.853891,0.052705,0.648599,0.967238,0.750812
2015-01-08,0.946574,0.715600,0.947135,0.899929,0.927798,0.355211,0.656076,0.395101,0.757294,0.560312,...,0.873241,0.994118,0.984171,0.916739,0.686123,0.560349,0.965053,0.900761,0.919060,0.871115
2015-01-09,0.568505,0.049834,0.190121,0.423166,0.163299,0.103537,0.607137,0.188148,0.318821,0.328078,...,0.519251,0.223012,0.108690,0.246675,0.083849,0.025381,0.197541,0.143619,0.177755,0.463034
2015-01-12,0.135397,0.479101,0.738297,0.332409,0.222127,0.107570,0.481049,0.587581,0.330468,0.265314,...,0.243780,0.166751,0.230887,0.501525,0.251893,0.698628,0.748344,0.227494,0.721871,0.090287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.834055,0.737016,0.632413,0.652496,0.495822,0.641206,0.688046,0.557480,0.515387,0.865227,...,0.836950,0.477872,0.817132,0.624697,0.727762,0.782660,0.358148,0.836621,0.979256,0.533666
2024-12-26,0.625625,0.310815,0.661002,0.372629,0.529149,0.710029,0.330317,0.314645,0.389582,0.230570,...,0.383004,0.660917,0.564878,0.509272,0.576803,0.274820,0.645950,0.552014,0.514067,0.529845
2024-12-27,0.134787,0.256757,0.369840,0.174834,0.351638,0.321902,0.495513,0.413751,0.238880,0.165104,...,0.363910,0.442348,0.437271,0.414662,0.237351,0.215032,0.447325,0.246108,0.139056,0.493820


In [100]:
from factor_copula import fit_static_clusters, select_n_groups, cluster_table

# --- Option 1 : fixer G et voir les groupes ---
result = fit_static_clusters(U, G=21, n_starts=10, seed=42)
print(cluster_table(result))

# --- Option 2 : sélection automatique de G par BIC ---
results = select_n_groups(U, G_range=range(3, 26), n_starts=10)
best_G = min(results, key=lambda g: results[g].bic)
print(cluster_table(results[best_G]))

    Group  Size   λ̃_M   λ̃_C  \
0       4    11  0.579  0.430   
1      13    11  0.692  0.447   
2       9    10  0.471  0.444   
3       6     7  0.486  0.000   
4      11     6  0.654  0.176   
5      20     6  0.509  0.552   
6      10     5  0.372  0.682   
7      21     5  0.596  0.274   
8       8     5  0.696  0.133   
9       5     4  0.469  0.497   
10     18     4  0.598  0.466   
11      2     3  0.499  0.475   
12     17     3  0.587  0.098   
13      3     3  0.529  0.699   
14      1     3  0.360  0.150   
15      7     2  0.641  0.590   
16     12     2  0.678  0.481   
17     14     2  0.761  0.094   
18     15     2  0.724  0.524   
19     16     2  0.616  0.000   
20     19     2  0.631  0.538   

                                              Members  
0   AAPL, ADBE, AMZN, AVGO, CRM, INTC, INTU, META,...  
1   AIG, AXP, BK, C, COF, GS, JPM, MET, SCHW, USB,...  
2   ABBV, AMGN, BMY, CVS, GILD, JNJ, LLY, MRK, PFE...  
3                 AMD, CHTR, GE, LMT, NFLX, SPG, 